# Stage 05: Collect RAG and memory router-training traces

**File:** `05_collect_training_traces_lane_00.ipynb`  
**Owner:** lane-00  
**Fixed recovery worker:** `stage-05-lane-00`  
**Success gate:** `TRACE_LANE_SEAL.json`

Execute one route-owned slice of the deployable train/calibration/validation matrix; sequential reference models are excluded from router fitting.

Run all four fixed stage-05 lane notebooks in parallel after stage 04. A stopped lane resumes by rerunning its same notebook.

This notebook is standalone for a fresh Kaggle session: it restores required
remote gates and the newest checksum-verified checkpoint from Hugging Face.
There is no team-roster notebook and there are no worker parameters to edit.
Never run this exact file in two live sessions. Rerunning the same file after a
stop is the supported resume path.


In [ ]:
# Fixed settings: do not edit. A lane file already has its permanent identity.
EXPERIMENT_ID = 'e2am-memrag-v3r1'
STAGE_ID = '05'
STAGE_NAME = 'collect_training_traces'
ROLE = 'lane'
LANE_ID = 'lane-00'
LANE_COUNT = 4
WORKER_ID = 'stage-05-lane-00'
NOTEBOOK_NAME = '05_collect_training_traces_lane_00.ipynb'

HF_REPO_ID = 'Shanmuk4622/E2AM-MemRAG-Traces'
HF_REPO_TYPE = 'dataset'
HF_REVISION = f'stage-{EXPERIMENT_ID}-{STAGE_ID}-{WORKER_ID}'
ARTIFACT_PREFIX = 'experiments/e2am-memrag-v3r1/stages/05/lane-00'
REQUIRED_GATES = ('04/coordinator/PILOT_ROUTE_FREEZE.json',)
OUTPUT_GATE = 'TRACE_LANE_SEAL.json'
SYNC_INTERVAL_SECONDS = 1200
WORK_ROOT = '/kaggle/working/e2am_memrag'
MODEL_CACHE_ROOT = WORK_ROOT + '/model-cache'
GPU_INDEX = 0

# The mask must happen before Torch, Transformers, NVML, or any CUDA-aware import.
import os
import sys

if 'torch' in sys.modules:
    raise RuntimeError(
        'Torch was imported before the one-GPU mask. Restart the kernel and Run All.'
    )
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES'] = str(GPU_INDEX)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Hugging Face reads these at import time. Keep the public model cache on the
# measured Kaggle working filesystem and turn a silent 0% transfer into a
# bounded, resumable HTTP download with explicit timeouts.
os.environ['HF_HOME'] = MODEL_CACHE_ROOT
os.environ['HF_HUB_CACHE'] = MODEL_CACHE_ROOT + '/hub'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '30'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
# The hf-xet worker can remain alive without advancing bytes on Kaggle. Use
# the bounded HTTP client instead; the verified runtime refreshes only the
# narrowly identified transient signed-blob 403 and preserves partial blobs.
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['E2AM_SYNC_INTERVAL_SECONDS'] = str(SYNC_INTERVAL_SECONDS)
os.environ['E2AM_HF_REPO_ID'] = HF_REPO_ID
print({
    'stage': STAGE_ID,
    'worker': WORKER_ID,
    'lane': LANE_ID,
    'sync_minutes': SYNC_INTERVAL_SECONDS // 60,
})


## Numbered Kaggle runbook

1. Open this notebook in Kaggle. Turn **Internet on** and select a **GPU** accelerator.
   A dual-T4 session is acceptable; the first code cell exposes only GPU 0.
2. Add a Kaggle Secret named `HF_TOKEN` with write access to
   `Shanmuk4622/E2AM-MemRAG-Traces`. Never paste the token into a cell.
3. Confirm every prerequisite gate below exists, then choose **Run All**.
   This is fixed `lane-00` of `4`; do not change its lane ID.
4. Leave the notebook running until it prints `STAGE_COMPLETE` and
   `REMOTE_CLOSURE_VERIFIED`. Dirty state is bundled every 20 minutes and after
   major units; clean intervals do not create Hub commits.
5. If you interrupt execution, wait for `SAFE_STOP_VERIFIED`. On a hard session
   loss, open this same notebook and Run All; only the smallest unsealed unit may replay.

### Required remote gates

- 04/coordinator/PILOT_ROUTE_FREEZE.json

### Work performed

- restore the frozen work plan and skip canonical unit IDs already completed
- collect query-only, charged-probe, retrieval, generation, and citation-guard costs
- give memory policies the same event stream and information budget
- record abstentions, failures, coverage, latency, GPU joules, and quality targets
- seal replay-bounded shards before each 20-minute or major remote receipt

### Expected artifacts under `experiments/e2am-memrag-v3r1/stages/05/lane-00/`

- matrix_work_plan.json
- results/shards/
- lane_metadata.json
- lane_export.zip
- TRACE_LANE_SEAL.json


## Restore the embedded verified runtime

This notebook contains the complete current `src/e2am_memrag`, `configs`,
`requirements-kaggle.txt`, and `pyproject.toml` runtime. The next cell verifies
the ZIP hash, exact member set, every member hash, and the source-tree hash before
atomically replacing `/kaggle/working/E2AM-MemRAG`. It keeps Kaggle's existing
Torch/CUDA packages and adds `src` to `sys.path` before importing the pipeline.


In [ ]:
# Deterministic runtime embedded by scripts/build_experiment_notebooks.py.
EMBEDDED_RUNTIME_B64 = (
    '...'
)
EXPECTED_RUNTIME_ARCHIVE_SHA256 = 'fba81b52d6978c56d4e0b87faa109e29a637fac9b778af21a43ecbd763b66a0c'
EXPECTED_RUNTIME_TREE_SHA256 = '3691c791374db5950eb0038a8f3f48b701a8f7302a6a8cddb80db3d497fc8954'
EXPECTED_RUNTIME_FILE_COUNT = 34
RUNTIME_MANIFEST_PATH = 'E2AM_RUNTIME_MANIFEST.json'
RUNTIME_ARCHIVE_MAX_BYTES = 67108864

import base64
import hashlib
import importlib
import io
import json
import re
import shutil
import stat
import subprocess
import tempfile
import zipfile
from pathlib import Path, PurePosixPath

def canonical_json(value):
    return json.dumps(
        value,
        sort_keys=True,
        separators=(',', ':'),
        ensure_ascii=False,
        allow_nan=False,
    )

archive_bytes = base64.b64decode(''.join(EMBEDDED_RUNTIME_B64), validate=True)
if len(archive_bytes) > RUNTIME_ARCHIVE_MAX_BYTES:
    raise RuntimeError('Embedded runtime exceeds its declared safety ceiling')
archive_sha256 = hashlib.sha256(archive_bytes).hexdigest()
if archive_sha256 != EXPECTED_RUNTIME_ARCHIVE_SHA256:
    raise RuntimeError('Embedded runtime archive SHA-256 mismatch')

verified_payloads = {}
with zipfile.ZipFile(io.BytesIO(archive_bytes), mode='r') as runtime_zip:
    infos = runtime_zip.infolist()
    names = [info.filename for info in infos]
    if len(names) != len(set(names)):
        raise RuntimeError('Embedded runtime ZIP contains duplicate member names')
    for info in infos:
        member = PurePosixPath(info.filename)
        if (
            member.is_absolute()
            or '..' in member.parts
            or '\\' in info.filename
            or info.filename.startswith('/')
        ):
            raise RuntimeError(f'Unsafe embedded runtime path: {info.filename!r}')
        unix_mode = info.external_attr >> 16
        if info.is_dir() or not stat.S_ISREG(unix_mode):
            raise RuntimeError(f'Non-regular runtime member: {info.filename!r}')
        if info.flag_bits & 0x1:
            raise RuntimeError(f'Encrypted runtime member is forbidden: {info.filename!r}')

    if RUNTIME_MANIFEST_PATH not in names:
        raise RuntimeError('Embedded runtime manifest is missing')
    manifest_bytes = runtime_zip.read(RUNTIME_MANIFEST_PATH)
    manifest = json.loads(manifest_bytes.decode('utf-8'))
    if set(manifest) != {
        'schema_version', 'source_tree_sha256', 'file_count', 'files'
    }:
        raise RuntimeError('Embedded runtime manifest schema is unexpected')
    if manifest['schema_version'] != 1:
        raise RuntimeError('Unsupported embedded runtime manifest version')
    entries = manifest['files']
    if not isinstance(entries, list) or len(entries) != EXPECTED_RUNTIME_FILE_COUNT:
        raise RuntimeError('Embedded runtime member count mismatch')
    if manifest['file_count'] != len(entries):
        raise RuntimeError('Embedded runtime manifest file_count mismatch')
    computed_tree_sha256 = hashlib.sha256(
        canonical_json(entries).encode('utf-8')
    ).hexdigest()
    if (
        manifest['source_tree_sha256'] != EXPECTED_RUNTIME_TREE_SHA256
        or computed_tree_sha256 != EXPECTED_RUNTIME_TREE_SHA256
    ):
        raise RuntimeError('Embedded runtime tree SHA-256 mismatch')

    expected_names = {RUNTIME_MANIFEST_PATH}
    total_uncompressed_bytes = len(manifest_bytes)
    for entry in entries:
        if not isinstance(entry, dict) or set(entry) != {'path', 'sha256', 'bytes'}:
            raise RuntimeError('Malformed embedded runtime file record')
        member_name = entry['path']
        if not isinstance(member_name, str) or member_name in expected_names:
            raise RuntimeError(f'Duplicate or invalid runtime member: {member_name!r}')
        if not isinstance(entry['bytes'], int) or entry['bytes'] < 0:
            raise RuntimeError(f'Invalid runtime member size: {member_name!r}')
        if (
            not isinstance(entry['sha256'], str)
            or len(entry['sha256']) != 64
            or any(character not in '0123456789abcdef' for character in entry['sha256'])
        ):
            raise RuntimeError(f'Invalid runtime member SHA-256: {member_name!r}')
        expected_names.add(member_name)
        total_uncompressed_bytes += entry['bytes']
        payload = runtime_zip.read(member_name)
        if len(payload) != entry['bytes']:
            raise RuntimeError(f'Runtime member size mismatch: {member_name}')
        if hashlib.sha256(payload).hexdigest() != entry['sha256']:
            raise RuntimeError(f'Runtime member SHA-256 mismatch: {member_name}')
        verified_payloads[member_name] = payload
    if set(names) != expected_names:
        extra = sorted(set(names) - expected_names)
        missing = sorted(expected_names - set(names))
        raise RuntimeError(
            f'Runtime ZIP member-set mismatch: extra={extra}, missing={missing}'
        )
    if total_uncompressed_bytes > RUNTIME_ARCHIVE_MAX_BYTES:
        raise RuntimeError('Embedded runtime uncompressed size exceeds safety ceiling')

project_root = Path('/kaggle/working/E2AM-MemRAG')
project_parent = project_root.parent
project_parent.mkdir(parents=True, exist_ok=True)
extraction_stage = Path(
    tempfile.mkdtemp(prefix='.e2am-runtime-', dir=str(project_parent))
)
backup_root = project_parent / '.E2AM-MemRAG.previous'
moved_existing = False
try:
    for member_name, payload in sorted(verified_payloads.items()):
        target = extraction_stage.joinpath(*PurePosixPath(member_name).parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open('xb') as handle:
            handle.write(payload)
            handle.flush()
            os.fsync(handle.fileno())
    manifest_target = extraction_stage / RUNTIME_MANIFEST_PATH
    with manifest_target.open('xb') as handle:
        handle.write(manifest_bytes)
        handle.flush()
        os.fsync(handle.fileno())

    for entry in entries:
        extracted = extraction_stage.joinpath(*PurePosixPath(entry['path']).parts)
        if extracted.stat().st_size != entry['bytes']:
            raise RuntimeError(f'Extracted runtime size mismatch: {entry["path"]}')
        if hashlib.sha256(extracted.read_bytes()).hexdigest() != entry['sha256']:
            raise RuntimeError(f'Extracted runtime SHA-256 mismatch: {entry["path"]}')
    if manifest_target.read_bytes() != manifest_bytes:
        raise RuntimeError('Extracted runtime manifest differs from embedded bytes')

    if backup_root.exists():
        shutil.rmtree(backup_root)
    if project_root.exists():
        os.replace(project_root, backup_root)
        moved_existing = True
    try:
        os.replace(extraction_stage, project_root)
    except BaseException:
        if moved_existing and backup_root.exists() and not project_root.exists():
            os.replace(backup_root, project_root)
        raise
    if backup_root.exists():
        shutil.rmtree(backup_root, ignore_errors=True)
except BaseException:
    if extraction_stage.exists():
        shutil.rmtree(extraction_stage, ignore_errors=True)
    raise

requirements_path = project_root / 'requirements-kaggle.txt'
install_requirements = []
skipped_torch_packages = []
for raw_line in requirements_path.read_text(encoding='utf-8').splitlines():
    requirement = raw_line.split('#', 1)[0].strip()
    if not requirement:
        continue
    if requirement.startswith('-'):
        raise RuntimeError(f'Unsupported requirements option: {requirement!r}')
    package_name = re.split(r'[<>=!~@\[\s]', requirement, maxsplit=1)[0]
    normalized_name = package_name.lower().replace('_', '-').replace('.', '-')
    if normalized_name in {'torch', 'torchvision', 'torchaudio'}:
        skipped_torch_packages.append(package_name)
        continue
    install_requirements.append(requirement)
if install_requirements:
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '--disable-pip-version-check',
        '--no-cache-dir',
        '--upgrade-strategy',
        'only-if-needed',
        '-q',
        *install_requirements,
    ])
if skipped_torch_packages:
    print('Preserved Kaggle Torch stack; skipped:', sorted(skipped_torch_packages))


project_src = str(project_root / 'src')
sys.path[:] = [project_src, *[item for item in sys.path if item != project_src]]
importlib.invalidate_caches()
os.environ['E2AM_SOURCE_TREE_SHA256'] = EXPECTED_RUNTIME_TREE_SHA256
print({
    'runtime': 'EMBEDDED_RUNTIME_VERIFIED',
    'project_root': str(project_root),
    'archive_sha256': archive_sha256,
    'tree_sha256': computed_tree_sha256,
    'file_count': len(entries),
})


In [ ]:
# Read the credential from Kaggle Secrets without printing or serializing it.
def load_hf_token():
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient

            token = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        raise RuntimeError(
            'HF_TOKEN is missing. Add it in Kaggle > Add-ons > Secrets, enable it, '
            'then restart and Run All.'
        )
    os.environ['HF_TOKEN'] = token
    return token

HF_TOKEN = load_hf_token()

# The previous cell verified and atomically restored this package before import.
try:
    import torch
    from e2am_memrag.experiment_pipeline import (
        StageRequest,
        finalize_stage,
        prepare_stage,
        run_stage,
        safe_stop_stage,
    )
except ModuleNotFoundError as error:
    raise RuntimeError(
        'The E2AM-MemRAG runtime is missing from this notebook release. '
        'Use the generated release notebook, not a copied cell fragment.'
    ) from error

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Enable a Kaggle GPU and restart the kernel.')
if torch.cuda.device_count() != 1:
    raise RuntimeError(
        f'Expected exactly one visible GPU after masking; got {torch.cuda.device_count()}.'
    )
gpu_name = torch.cuda.get_device_name(0)
if 'T4' not in gpu_name.upper():
    raise RuntimeError(f'Primary experiment requires a T4; visible device is {gpu_name!r}.')
print({'visible_gpu_count': 1, 'gpu_name': gpu_name, 'hf_token_loaded': True})


## Preflight and restore

This cell is read/verify-first. It validates the remote prerequisites and lane
binding before it can append data. A corrupt or unsealed checkpoint is ignored in
favor of the previous verified checkpoint.


In [ ]:
REQUEST = StageRequest(
    experiment_id=EXPERIMENT_ID,
    stage_id=STAGE_ID,
    stage_name=STAGE_NAME,
    role=ROLE,
    worker_id=WORKER_ID,
    lane_id=LANE_ID,
    lane_count=LANE_COUNT,
    notebook_name=NOTEBOOK_NAME,
    hf_repo_id=HF_REPO_ID,
    hf_repo_type=HF_REPO_TYPE,
    hf_revision=HF_REVISION,
    artifact_prefix=ARTIFACT_PREFIX,
    required_gates=REQUIRED_GATES,
    output_gate=OUTPUT_GATE,
    sync_interval_seconds=SYNC_INTERVAL_SECONDS,
    work_root=WORK_ROOT,
    stage_work_items=('restore the frozen work plan and skip canonical unit IDs already completed', 'collect query-only, charged-probe, retrieval, generation, and citation-guard costs', 'give memory policies the same event stream and information budget', 'record abstentions, failures, coverage, latency, GPU joules, and quality targets', 'seal replay-bounded shards before each 20-minute or major remote receipt'),
)

# prepare_stage authenticates once, verifies prerequisite gates at pinned commits,
# measures storage, restores the latest valid closure/checkpoint, and rejects a
# changed spec/environment/lane owner before scientific work begins.
RUNTIME = prepare_stage(REQUEST, hf_token=HF_TOKEN)
PREPARE_REPORT = RUNTIME.prepare_report
if not PREPARE_REPORT.get('go', False):
    raise RuntimeError(f'PREPARE_NO_GO: {PREPARE_REPORT.get("reason", "unknown")}')
print('PREPARE_GO')
print({
    'stage': STAGE_ID,
    'worker': WORKER_ID,
    'restored': PREPARE_REPORT.get('restored', False),
    'completed_units': PREPARE_REPORT.get('completed_units', 0),
})


## Execute resumable work

The runtime uses immutable logical unit IDs. Physical execution is at least once;
aggregation accepts exactly one canonical artifact for each unit. A manual interrupt
requests a checkpoint and immediate verified Hub closure at the next safe boundary.


In [ ]:
# run_stage owns deterministic unit assignment and replay-bounded result checkpoints,
# 20-minute dirty sync, major-boundary sync, and upload-free energy blocks.
STAGE_RESULT = None
try:
    STAGE_RESULT = run_stage(RUNTIME)
except KeyboardInterrupt:
    stop_report = safe_stop_stage(RUNTIME, reason='keyboard_interrupt')
    if stop_report.get('remote_verified', False):
        print('SAFE_STOP_VERIFIED')
    else:
        print('SAFE_STOP_INCOMPLETE:', stop_report.get('local_resume_path'))
    raise
except BaseException:
    # Preserve the original exception while making a bounded best-effort closure.
    try:
        stop_report = safe_stop_stage(RUNTIME, reason='stage_exception')
        status = (
            'SAFE_STOP_VERIFIED'
            if stop_report.get('remote_verified')
            else 'SAFE_STOP_INCOMPLETE'
        )
        print(status)
    except Exception as stop_error:
        print('SAFE_STOP_INCOMPLETE:', type(stop_error).__name__)
    raise

if STAGE_RESULT is None:
    raise RuntimeError('Stage returned no result; finalization is forbidden.')
print({
    'new_units': STAGE_RESULT.get('new_units', 0),
    'reused_units': STAGE_RESULT.get('reused_units', 0),
    'failed_units': STAGE_RESULT.get('failed_units', 0),
})


## Verify and publish the stage gate

Normal completion forces a remote closure even if the 20-minute timer is not due.
The gate and pointer are published only after every referenced artifact exists.


In [ ]:
# Finalization verifies every dependency, commits the stage gate with its closure,
# pins the returned SHA, and downloads the receipt/seal/pointer for checksum proof.
FINAL_REPORT = finalize_stage(RUNTIME, STAGE_RESULT)
if not FINAL_REPORT.get('remote_verified', False):
    raise RuntimeError(
        'FINAL_NO_GO: local artifacts exist but the remote closure is not verified. '
        f'Rerun this same notebook to resume: {FINAL_REPORT.get("local_resume_path")}'
    )
if FINAL_REPORT.get('output_gate') != OUTPUT_GATE:
    raise RuntimeError('FINAL_NO_GO: the verified gate does not match this notebook.')

print('REMOTE_CLOSURE_VERIFIED')
print('STAGE_COMPLETE:', STAGE_ID, OUTPUT_GATE)
print({
    'revision': HF_REVISION,
    'commit_sha': FINAL_REPORT.get('commit_sha'),
    'artifact_prefix': ARTIFACT_PREFIX,
    'output_gate': OUTPUT_GATE,
})


## Troubleshooting and exact resume

- **`HF_TOKEN is missing`**: enable the Kaggle Secret named `HF_TOKEN`; do not paste
  a token into the notebook.
- **401/403**: the uploader stops repeated attempts. Correct repository write access,
  then rerun `05_collect_training_traces_lane_00.ipynb`.
- **429/rate limit**: sealed work stays local while the runtime honors `Retry-After`
  and exponential backoff. The normal dirty-state target is 1,200 seconds.
- **Torch imported before mask / wrong GPU count**: restart the kernel and use Run All.
  Do not execute setup cells out of order.
- **OOM or unsafe VRAM**: the declared route is recorded as failed. The notebook never
  silently changes batch size, precision, context, or device placement.
- **Storage gate**: new work stops before the emergency reserve is consumed. Only
  reproducible caches may be deleted after their source/checksum is recorded.
- **Interrupted upload or lost acknowledgement**: rerun this same fixed notebook.
  Recovery checks deterministic receipts and pinned remote hashes before retrying.
- **Hard Kaggle termination**: cleanup cannot run. The next Run All restores the latest
  verified closure and replays at most the smallest unsealed unit.
- **Prerequisite gate missing**: finish the earlier numbered notebook(s); never bypass
  the gate or copy artifacts between stage prefixes manually.

A run is complete only after both `REMOTE_CLOSURE_VERIFIED` and `STAGE_COMPLETE`
appear. Keep the Kaggle session open if only `SAFE_STOP_INCOMPLETE` is shown.
